# Análise Exploratória - MICRODADOS INEP 2024
## Censo da Educação Superior - Cadastro de Cursos

**Objetivo**: Explorar e gerar insights a partir da base de microdados do INEP 2024

**Estrutura**:
- Exploração inicial da base (430 MB)
- Análise descritiva por região, modalidade, grau acadêmico
- Distribuição de matrículas, ingressantes e vagas
- Trends por tipo de organização acadêmica e categoria administrativa
- Exportação de resultados em formato visual

In [12]:
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Configuracoes
ROOT_DIR = Path.cwd()
MAIN_CSV_PATH = ROOT_DIR / 'MICRODADOS_CADASTRO_CURSOS_2024.CSV'
IES_CSV_PATH = ROOT_DIR / 'MICRODADOS_ED_SUP_IES_2024.CSV'
SLIM_CSV_PATH = ROOT_DIR / 'MICRODADOS_CADASTRO_CURSOS_2024_SLIM.csv'
OUTPUT_DIR = ROOT_DIR / 'outputs_inep'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Arquivo principal: {MAIN_CSV_PATH}')
print(f'Existe principal: {MAIN_CSV_PATH.exists()}')
print(f'Tamanho principal: {MAIN_CSV_PATH.stat().st_size / (1024**2):.2f} MB' if MAIN_CSV_PATH.exists() else 'Arquivo principal nao encontrado')
print(f'Arquivo IES: {IES_CSV_PATH}')
print(f'Existe IES: {IES_CSV_PATH.exists()}')
print(f'Arquivo reduzido: {SLIM_CSV_PATH}')
print(f'Existe reduzido: {SLIM_CSV_PATH.exists()}')

Arquivo principal: c:\Users\alian\source\repos\VISAGIO\MICRODADOS_CADASTRO_CURSOS_2024.CSV
Existe principal: True
Tamanho principal: 431.87 MB
Arquivo IES: c:\Users\alian\source\repos\VISAGIO\MICRODADOS_ED_SUP_IES_2024.CSV
Existe IES: True
Arquivo reduzido: c:\Users\alian\source\repos\VISAGIO\MICRODADOS_CADASTRO_CURSOS_2024_SLIM.csv
Existe reduzido: True


## 1. Carregamento Otimizado da Base

Estratégia:
- Ler apenas colunas essenciais para reduzir uso de memória
- Definir tipos de dados corretos na leitura
- Usar sep=';' (ponto-e-vírgula)

In [13]:
# Primeira pass: ler apenas para inspecionar nomes das colunas do arquivo principal
print('Lendo cabecalho do arquivo principal...')
df_header = pd.read_csv(MAIN_CSV_PATH, sep=';', nrows=0, encoding='latin-1')
print(f'Total de colunas no principal: {len(df_header.columns)}')
print('\nPrimeiras 10 colunas:')
for i, col in enumerate(df_header.columns[:10]):
    print(f'  {i}: {col}')

Lendo cabecalho do arquivo principal...
Total de colunas no principal: 223

Primeiras 10 colunas:
  0: NU_ANO_CENSO
  1: NO_REGIAO
  2: CO_REGIAO
  3: NO_UF
  4: SG_UF
  5: CO_UF
  6: NO_MUNICIPIO
  7: CO_MUNICIPIO
  8: IN_CAPITAL
  9: TP_DIMENSAO


In [14]:
# Colunas solicitadas no dicionario (TP_REDE opcional)
colunas_dicionario = [
    'NU_ANO_CENSO',
    'SG_UF',
    'NO_REGIAO',
    'CO_IES',
    'TP_REDE',
    'NO_CINE_ROTULO',
    'NO_CINE_AREA_GERAL',
    'TP_MODALIDADE_ENSINO',
    'QT_CURSO',
    'QT_VG_TOTAL',
    'QT_VG_TOTAL_EAD',
    'QT_ING',
    'QT_ING_FEM',
    'QT_ING_MASC',
    'QT_MAT',
    'QT_MAT_FEM',
    'QT_MAT_MASC',
]

# Colunas extras para manter as analises ja existentes do notebook e enriquecer com a base de IES
colunas_extras = [
    'TP_ORGANIZACAO_ACADEMICA',
    'TP_CATEGORIA_ADMINISTRATIVA',
    'QT_CONC',
    'NO_IES',
    'SG_IES',
]

colunas_alvo = list(dict.fromkeys(colunas_dicionario + colunas_extras))
colunas_disponiveis = [c for c in colunas_alvo if c in df_header.columns]
colunas_ausentes = [c for c in colunas_dicionario if c not in df_header.columns]

print(f'Colunas alvo: {len(colunas_alvo)}')
print(f'Colunas disponiveis no principal: {len(colunas_disponiveis)}')
if colunas_ausentes:
    print(f'Colunas do dicionario ausentes (serao ignoradas): {colunas_ausentes}')

# Se o arquivo reduzido nao existir ou estiver desatualizado, cria a partir do principal + base de IES
slim_required_cols = list(dict.fromkeys(colunas_disponiveis + ['NO_IES', 'SG_IES']))
slim_needs_rebuild = True
if SLIM_CSV_PATH.exists():
    try:
        df_slim_header = pd.read_csv(SLIM_CSV_PATH, sep=';', nrows=0, encoding='utf-8-sig')
        slim_needs_rebuild = not all(c in df_slim_header.columns for c in slim_required_cols)
    except Exception:
        slim_needs_rebuild = True

if slim_needs_rebuild:
    print('\nCriando/atualizando arquivo reduzido para consultas mais rapidas...')
    df_main_slim = pd.read_csv(
        MAIN_CSV_PATH,
        sep=';',
        encoding='latin-1',
        usecols=[c for c in colunas_disponiveis if c not in ['NO_IES', 'SG_IES']],
        low_memory=False,
        na_values=['', 'null', 'NULL'],
    )

    if IES_CSV_PATH.exists():
        df_ies = pd.read_csv(
            IES_CSV_PATH,
            sep=';',
            encoding='latin-1',
            usecols=['CO_IES', 'NO_IES', 'SG_IES'],
            low_memory=False,
            na_values=['', 'null', 'NULL'],
        )
        df_ies = df_ies.drop_duplicates(subset=['CO_IES']).copy()
        df_main_slim = df_main_slim.merge(df_ies, on='CO_IES', how='left')
    else:
        print('Aviso: base MICRODADOS_ED_SUP_IES_2024.CSV nao encontrada. NO_IES e SG_IES ficarao vazios.')
        if 'NO_IES' not in df_main_slim.columns:
            df_main_slim['NO_IES'] = pd.NA
        if 'SG_IES' not in df_main_slim.columns:
            df_main_slim['SG_IES'] = pd.NA

    # Reordena colunas para manter padrao previsivel
    final_cols = [c for c in slim_required_cols if c in df_main_slim.columns]
    df_main_slim = df_main_slim[final_cols].copy()
    df_main_slim.to_csv(SLIM_CSV_PATH, sep=';', index=False, encoding='utf-8-sig')
    print(f'Arquivo reduzido criado/atualizado: {SLIM_CSV_PATH}')

# Definir dtypes para otimizacao de memoria
# Observacao: algumas colunas podem vir com valores faltantes; por isso usamos tipos tolerantes.
dtype_map = {
    'NU_ANO_CENSO': 'Int64',
    'SG_UF': 'string',
    'NO_REGIAO': 'string',
    'CO_IES': 'Int64',
    'TP_REDE': 'Int64',
    'NO_CINE_ROTULO': 'string',
    'NO_CINE_AREA_GERAL': 'string',
    'TP_MODALIDADE_ENSINO': 'Int64',
    'TP_ORGANIZACAO_ACADEMICA': 'Int64',
    'TP_CATEGORIA_ADMINISTRATIVA': 'Int64',
    'QT_CURSO': 'Int64',
    'QT_VG_TOTAL': 'Int64',
    'QT_VG_TOTAL_EAD': 'Int64',
    'QT_ING': 'Int64',
    'QT_ING_FEM': 'Int64',
    'QT_ING_MASC': 'Int64',
    'QT_MAT': 'Int64',
    'QT_MAT_FEM': 'Int64',
    'QT_MAT_MASC': 'Int64',
    'QT_CONC': 'Int64',
    'NO_IES': 'string',
    'SG_IES': 'string',
}

print('\nCarregando base reduzida...')
df_inep = pd.read_csv(
    SLIM_CSV_PATH,
    sep=';',
    encoding='utf-8-sig',
    dtype={k: v for k, v in dtype_map.items() if k in slim_required_cols},
    low_memory=False,
)

# Materializa lista final de colunas para a secao dinamica
COLUNAS_ANALISE_DICIONARIO = [c for c in colunas_dicionario if c in df_inep.columns]

print(f'\n✓ Base reduzida carregada com sucesso!')
print(f'Shape: {df_inep.shape}')
print(f'Memoria utilizada: {df_inep.memory_usage(deep=True).sum() / (1024**2):.2f} MB')
print(f'Colunas de analise (dicionario) disponiveis: {COLUNAS_ANALISE_DICIONARIO}')
print(f'Colunas agregadas da base IES disponiveis: {[c for c in ["NO_IES", "SG_IES"] if c in df_inep.columns]}')

Colunas alvo: 22
Colunas disponiveis no principal: 20

Carregando base reduzida...

✓ Base reduzida carregada com sucesso!
Shape: (720349, 22)
Memoria utilizada: 440.71 MB
Colunas de analise (dicionario) disponiveis: ['NU_ANO_CENSO', 'SG_UF', 'NO_REGIAO', 'CO_IES', 'TP_REDE', 'NO_CINE_ROTULO', 'NO_CINE_AREA_GERAL', 'TP_MODALIDADE_ENSINO', 'QT_CURSO', 'QT_VG_TOTAL', 'QT_VG_TOTAL_EAD', 'QT_ING', 'QT_ING_FEM', 'QT_ING_MASC', 'QT_MAT', 'QT_MAT_FEM', 'QT_MAT_MASC']
Colunas agregadas da base IES disponiveis: ['NO_IES', 'SG_IES']


## 1.1 Teste de Destaque V-Educa no INEP

Valida a regra usada no app para separar visualmente os grupos da V-Educa:

- `NO_IES` presente na base V-Educa
- `TP_MODALIDADE_ENSINO = 1` -> `VEDUCA Presencial`
- `TP_MODALIDADE_ENSINO = 2` -> `VEDUCA EAD`

O objetivo e testar a classificacao sem cortar registros do INEP.

In [21]:
# Teste da identificacao V-Educa no INEP por NO_IES + TP_MODALIDADE_ENSINO

# Opcional: se a base V-Educa nao tiver coluna de IES, preencha manualmente alguns nomes de IES abaixo.
ies_veduca_manual = [
    # 'UNIVERSIDADE EXEMPLO',
    # 'FACULDADE EXEMPLO',
]

# 1) Carregar IES da V-Educa a partir do base.xlsx (extraindo o bloco principal)
base_excel_path = ROOT_DIR / 'base.xlsx'

def extrair_bloco_principal_excel(xlsx_path, sheet_name):
    raw = pd.read_excel(xlsx_path, sheet_name=sheet_name, header=None, engine='openpyxl')
    non_na_counts = raw.notna().sum(axis=1)
    candidate_rows = non_na_counts[non_na_counts >= 3].index.tolist()
    if not candidate_rows:
        return pd.DataFrame()

    blocos = []
    start = candidate_rows[0]
    prev = candidate_rows[0]
    for row in candidate_rows[1:]:
        if row == prev + 1:
            prev = row
        else:
            blocos.append((start, prev))
            start = row
            prev = row
    blocos.append((start, prev))

    block_start, block_end = max(blocos, key=lambda x: x[1] - x[0])
    block = raw.iloc[block_start:block_end + 1].copy()

    header_candidates = block.head(3)
    best_idx = 0
    best_score = -1.0
    for i in range(len(header_candidates)):
        row = header_candidates.iloc[i].astype(str)
        cleaned = row.replace('nan', '').str.strip()
        non_empty = int((cleaned != '').sum())
        unique = int(cleaned[cleaned != ''].nunique())
        score = non_empty + unique * 0.5
        if score > best_score:
            best_score = score
            best_idx = i

    header = header_candidates.iloc[best_idx].astype(str).replace('nan', '').str.strip()
    renamed_cols = []
    seen = {}
    for j, col in enumerate(header):
        name = col if col else f'col_{j}'
        if name in seen:
            seen[name] += 1
            name = f'{name}_{seen[name]}'
        else:
            seen[name] = 0
        renamed_cols.append(name)

    data = block.iloc[best_idx + 1:].copy()
    data.columns = renamed_cols
    data = data.dropna(how='all')
    data = data.loc[:, data.notna().any(axis=0)]
    return data

def encontrar_coluna_token(df_cols, token):
    token_up = token.upper()
    for c in df_cols:
        if token_up in str(c).upper():
            return c
    return None

def detectar_coluna_ies_fallback(df_cols):
    # Prioriza nomes textuais contendo IES e evita codigos como CO_IES
    candidatos = [c for c in df_cols if 'IES' in str(c).upper()]
    for c in candidatos:
        c_up = str(c).upper()
        if 'CO_IES' in c_up:
            continue
        if 'NO_' in c_up or 'NOME' in c_up or c_up.strip() == 'IES':
            return c
    return candidatos[0] if candidatos else None

alunos_veduca = extrair_bloco_principal_excel(base_excel_path, 'Alunos V-Educa')
col_no_ies_veduca = encontrar_coluna_token(alunos_veduca.columns, 'NO_IES')
if col_no_ies_veduca is None:
    col_no_ies_veduca = detectar_coluna_ies_fallback(alunos_veduca.columns)

ies_veduca = set()
if col_no_ies_veduca is not None:
    print(f'Coluna usada como IES na V-Educa: {col_no_ies_veduca}')
    ies_veduca = set(alunos_veduca[col_no_ies_veduca].dropna().astype(str).str.strip())
    ies_veduca = {v for v in ies_veduca if v}
elif ies_veduca_manual:
    ies_veduca = {str(v).strip() for v in ies_veduca_manual if str(v).strip()}
    print('Usando lista manual de IES para teste (ies_veduca_manual).')
else:
    print('Coluna de IES nao encontrada na base V-Educa extraida e lista manual vazia.')
    print('Colunas disponiveis:')
    print(list(alunos_veduca.columns))

if ies_veduca:
    print(f'IES unicas para match V-Educa: {len(ies_veduca):,}'.replace(',', '.'))

    # 2) Aplicar classificacao sem remover linhas do INEP
    df_teste = df_inep.copy()
    df_teste['__modal_label__'] = df_teste['TP_MODALIDADE_ENSINO'].map({1: 'Presencial', 2: 'Curso a distancia'})
    df_teste['__modal_label__'] = df_teste['__modal_label__'].fillna('Outros')

    in_veduca = df_teste['NO_IES'].astype(str).str.strip().isin(ies_veduca)
    modalidade_num = pd.to_numeric(df_teste['TP_MODALIDADE_ENSINO'], errors='coerce')

    df_teste.loc[in_veduca & (modalidade_num == 1), '__modal_label__'] = 'VEDUCA Presencial'
    df_teste.loc[in_veduca & (modalidade_num == 2), '__modal_label__'] = 'VEDUCA EAD'

    # 3) Resumo de validacao
    resumo_grupos = (
        df_teste.groupby('__modal_label__', observed=True)
        .agg(
            registros=('CO_IES', 'size'),
            qt_mat=('QT_MAT', 'sum'),
            qt_ing=('QT_ING', 'sum'),
        )
        .sort_values('qt_mat', ascending=False)
    )

    print('\nResumo por grupo de modalidade com destaque V-Educa:')
    print(resumo_grupos)

    # 4) Grafico para teste visual (tons de laranja para V-Educa)
    ordem = ['Presencial', 'Curso a distancia', 'VEDUCA Presencial', 'VEDUCA EAD', 'Outros']
    resumo_plot = resumo_grupos.reset_index().rename(columns={'__modal_label__': 'grupo'})
    resumo_plot['grupo'] = pd.Categorical(resumo_plot['grupo'], categories=ordem, ordered=True)
    resumo_plot = resumo_plot.sort_values('grupo')

    color_map = {
        'Presencial': '#2ca02c',
        'Curso a distancia': '#1f77b4',
        'VEDUCA Presencial': '#ff8c42',
        'VEDUCA EAD': '#ffb26b',
        'Outros': '#999999',
    }

    fig_teste = px.bar(
        resumo_plot,
        x='grupo',
        y='qt_mat',
        text='qt_mat',
        title='Teste de destaque V-Educa no INEP (MATRICULAS)',
        labels={'grupo': 'Grupo', 'qt_mat': 'Matriculados (QT_MAT)'},
        color='grupo',
        color_discrete_map=color_map,
    )
    fig_teste.update_traces(texttemplate='%{text:,.0f}', textposition='outside')
    fig_teste.update_layout(showlegend=False, height=520)
    fig_teste.show()

Coluna de IES nao encontrada na base V-Educa extraida e lista manual vazia.
Colunas disponiveis:
['ANO', 'UF', 'AREA', 'CURSO', 'MODALIDADE', 'TICKET MÉDIO', 'INGRESSANTES', 'MATRICULADOS', 'NO_CINE_AREA_GERAL']


## 2. Exploração Inicial

Análise estrutural dos dados carregados

In [4]:
print("ESTRUTURA DA BASE")
print("=" * 80)
print(df_inep.info())
print("\n" + "=" * 80)
print(f"\nValores faltantes (%):")
missing_pct = (df_inep.isna().sum() / len(df_inep) * 100).sort_values(ascending=False)
print(missing_pct[missing_pct > 0].head(10))

ESTRUTURA DA BASE
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 720349 entries, 0 to 720348
Data columns (total 22 columns):
 #   Column                       Non-Null Count   Dtype 
---  ------                       --------------   ----- 
 0   NU_ANO_CENSO                 720349 non-null  Int64 
 1   SG_UF                        708571 non-null  string
 2   NO_REGIAO                    708571 non-null  string
 3   CO_IES                       720349 non-null  Int64 
 4   TP_REDE                      720349 non-null  Int64 
 5   NO_CINE_ROTULO               720349 non-null  string
 6   NO_CINE_AREA_GERAL           720349 non-null  string
 7   TP_MODALIDADE_ENSINO         720349 non-null  Int64 
 8   QT_CURSO                     720349 non-null  Int64 
 9   QT_VG_TOTAL                  720349 non-null  Int64 
 10  QT_VG_TOTAL_EAD              720349 non-null  Int64 
 11  QT_ING                       720349 non-null  Int64 
 12  QT_ING_FEM                   720349 non-null  Int64 
 

In [5]:
print("\nDISTRIBUIÇÃO POR MODALIDADE")
print("=" * 80)
print(df_inep['TP_MODALIDADE_ENSINO'].value_counts())

print("\n\nDISTRIBUIÇÃO POR REGIÃO")
print("=" * 80)
print(df_inep['NO_REGIAO'].value_counts())

print("\n\nDISTRIBUIÇÃO POR TIPO DE ORGANIZAÇÃO ACADÊMICA")
print("=" * 80)
print(df_inep['TP_ORGANIZACAO_ACADEMICA'].value_counts())

print("\n\nDISTRIBUIÇÃO POR CATEGORIA ADMINISTRATIVA")
print("=" * 80)
print(df_inep['TP_CATEGORIA_ADMINISTRATIVA'].value_counts())


DISTRIBUIÇÃO POR MODALIDADE
TP_MODALIDADE_ENSINO
2    685525
1     34824
Name: count, dtype: Int64


DISTRIBUIÇÃO POR REGIÃO
NO_REGIAO
Sudeste         291176
Sul             157232
Nordeste        142672
Centro-Oeste     60041
Norte            57450
Name: count, dtype: Int64


DISTRIBUIÇÃO POR TIPO DE ORGANIZAÇÃO ACADÊMICA
TP_ORGANIZACAO_ACADEMICA
2    370724
1    323102
3     23985
4      2474
5        64
Name: count, dtype: Int64


DISTRIBUIÇÃO POR CATEGORIA ADMINISTRATIVA
TP_CATEGORIA_ADMINISTRATIVA
4    638763
5     61435
1     10352
2      8608
7       936
3       255
Name: count, dtype: Int64


In [6]:
print("\nESTATÍSTICAS PRINCIPAIS")
print("=" * 80)
print(f"Total de Cursos: {df_inep['QT_CURSO'].sum():,}")
print(f"Total de Vagas (QT_VG_TOTAL): {df_inep['QT_VG_TOTAL'].sum():,}")
print(f"Total de Ingressantes (QT_ING): {df_inep['QT_ING'].sum():,}")
print(f"Total de Matriculados (QT_MAT): {df_inep['QT_MAT'].sum():,}")
print(f"Total de Concluintes (QT_CONC): {df_inep['QT_CONC'].sum():,}")

print("\n\nESTATÍSTICAS DESCRITIVAS")
print("=" * 80)
stats_cols = ['QT_CURSO', 'QT_VG_TOTAL', 'QT_ING', 'QT_MAT', 'QT_CONC']
print(df_inep[stats_cols].describe())


ESTATÍSTICAS PRINCIPAIS
Total de Cursos: 45,776
Total de Vagas (QT_VG_TOTAL): 23,665,419
Total de Ingressantes (QT_ING): 5,010,613
Total de Matriculados (QT_MAT): 10,227,266
Total de Concluintes (QT_CONC): 1,333,988


ESTATÍSTICAS DESCRITIVAS
       QT_CURSO  QT_VG_TOTAL     QT_ING     QT_MAT    QT_CONC
count  720349.0     720349.0   720349.0   720349.0   720349.0
mean   0.063547    32.852713   6.955813  14.197654   1.851863
std    0.243944    782.10785  32.298981  71.467216  11.021008
min         0.0          0.0        0.0        0.0        0.0
25%         0.0          0.0        0.0        1.0        0.0
50%         0.0          0.0        1.0        1.0        0.0
75%         0.0          0.0        3.0        5.0        1.0
max         1.0     100485.0     5677.0    13985.0     1549.0


## 3. Análises por Região

Distribuição de matrículas, ingressantes e vagas por região

In [7]:
# Agregação por região
regiao_agg = df_inep.groupby('NO_REGIAO', observed=True).agg({
    'QT_CURSO': 'sum',
    'QT_VG_TOTAL': 'sum',
    'QT_ING': 'sum',
    'QT_MAT': 'sum',
    'QT_CONC': 'sum'
}).sort_values('QT_MAT', ascending=False)

print("MATRÍCULAS POR REGIÃO")
print("=" * 80)
print(regiao_agg)

# Gráfico por modalidade e região
modalidade_regiao = df_inep.groupby(['NO_REGIAO', 'TP_MODALIDADE_ENSINO'], observed=True)['QT_MAT'].sum().reset_index()

fig = px.bar(
    modalidade_regiao,
    x='NO_REGIAO',
    y='QT_MAT',
    color='TP_MODALIDADE_ENSINO',
    title='Matrículas por Região e Modalidade',
    labels={'QT_MAT': 'Matrículas', 'NO_REGIAO': 'Região', 'TP_MODALIDADE_ENSINO': 'Modalidade'},
    barmode='stack'
)
fig.show()

MATRÍCULAS POR REGIÃO
              QT_CURSO  QT_VG_TOTAL   QT_ING   QT_MAT  QT_CONC
NO_REGIAO                                                     
Sudeste          14366      2309727  2337201  4518942   627003
Nordeste          7526      1227869   944841  2186521   263345
Sul               6389       621655   882203  1750161   231226
Centro-Oeste      3409       451417   460760   905093   115081
Norte             2789       464478   384111   863969    96860


## 4. Análise por Modalidade de Ensino

Presencial, EAD e Híbrida - Comparação de métricas

In [8]:
modalidade_agg = df_inep.groupby('TP_MODALIDADE_ENSINO', observed=True).agg({
    'QT_CURSO': 'sum',
    'QT_VG_TOTAL': 'sum',
    'QT_ING': 'sum',
    'QT_MAT': 'sum',
    'QT_CONC': 'sum'
}).sort_values('QT_MAT', ascending=False)

print("ESTATÍSTICAS POR MODALIDADE")
print("=" * 80)
print(modalidade_agg)
print("\n% de Matrículas:")
print((modalidade_agg['QT_MAT'] / modalidade_agg['QT_MAT'].sum() * 100).round(2))

# Gráfico pizza de distribuição por modalidade
fig_pizza = go.Figure(data=[go.Pie(
    labels=modalidade_agg.index,
    values=modalidade_agg['QT_MAT'],
    textposition='auto',
    textinfo='label+percent'
)])
fig_pizza.update_layout(title='Distribuição de Matrículas por Modalidade')
fig_pizza.show()

ESTATÍSTICAS POR MODALIDADE
                      QT_CURSO  QT_VG_TOTAL   QT_ING   QT_MAT  QT_CONC
TP_MODALIDADE_ENSINO                                                  
2                        11297     18590273  3347573  5189391   604742
1                        34479      5075146  1663040  5037875   729246

% de Matrículas:
TP_MODALIDADE_ENSINO
2    50.74
1    49.26
Name: QT_MAT, dtype: Float64


## 5. Análise por Categoria Administrativa

Pública x Privada (Lucrativa e Filantrópica)

In [9]:
categoria_agg = df_inep.groupby('TP_CATEGORIA_ADMINISTRATIVA', observed=True).agg({
    'QT_CURSO': 'sum',
    'QT_VG_TOTAL': 'sum',
    'QT_ING': 'sum',
    'QT_MAT': 'sum',
    'QT_CONC': 'sum'
}).sort_values('QT_MAT', ascending=False)

print("ESTATÍSTICAS POR CATEGORIA ADMINISTRATIVA")
print("=" * 80)
print(categoria_agg)
print("\n% de Matrículas:")
print((categoria_agg['QT_MAT'] / categoria_agg['QT_MAT'].sum() * 100).round(2))

ESTATÍSTICAS POR CATEGORIA ADMINISTRATIVA
                             QT_CURSO  QT_VG_TOTAL   QT_ING   QT_MAT  QT_CONC
TP_CATEGORIA_ADMINISTRATIVA                                                  
4                               22363     18315459  3714283  6372474   805313
5                               11932      4366942   721000  1789725   272168
1                                7124       586246   345453  1317548   157067
2                                3701       296641   201760   666980    88129
7                                 416        75669    16064    45400     6986
3                                 240        24462    12053    35139     4325

% de Matrículas:
TP_CATEGORIA_ADMINISTRATIVA
4    62.31
5     17.5
1    12.88
2     6.52
7     0.44
3     0.34
Name: QT_MAT, dtype: Float64


## 6. Resumo Executivo e Exportação

Consolidação dos principais insights

In [10]:
# Criar resumo executivo
resumo = f"""
RESUMO EXECUTIVO - MICRODADOS INEP 2024
{'='*80}

Data: {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}

MÉTRICAS GLOBAIS
{'-'*80}
Total de Registros: {len(df_inep):,}
Total de Cursos: {df_inep['QT_CURSO'].sum():,}
Total de Vagas: {df_inep['QT_VG_TOTAL'].sum():,}
Total de Ingressantes: {df_inep['QT_ING'].sum():,}
Total de Matriculados: {df_inep['QT_MAT'].sum():,}
Total de Concluintes: {df_inep['QT_CONC'].sum():,}

DISTRIBUIÇÃO POR MODALIDADE
{'-'*80}
{modalidade_agg.to_string()}

DISTRIBUIÇÃO POR REGIÃO
{'-'*80}
{regiao_agg.to_string()}

DISTRIBUIÇÃO POR CATEGORIA ADMINISTRATIVA
{'-'*80}
{categoria_agg.to_string()}
"""

print(resumo)

# Salvar resumo em arquivo
resumo_path = OUTPUT_DIR / f"resumo_executivo_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
with open(resumo_path, 'w', encoding='utf-8') as f:
    f.write(resumo)
print(f"\n✓ Resumo salvo em: {resumo_path}")

# Exportar agregações
regiao_agg.to_csv(OUTPUT_DIR / 'analise_por_regiao.csv', encoding='utf-8-sig')
modalidade_agg.to_csv(OUTPUT_DIR / 'analise_por_modalidade.csv', encoding='utf-8-sig')
categoria_agg.to_csv(OUTPUT_DIR / 'analise_por_categoria_administrativa.csv', encoding='utf-8-sig')
print("✓ Agregações exportadas em CSV")


RESUMO EXECUTIVO - MICRODADOS INEP 2024

Data: 08/04/2026 15:11:09

MÉTRICAS GLOBAIS
--------------------------------------------------------------------------------
Total de Registros: 720,349
Total de Cursos: 45,776
Total de Vagas: 23,665,419
Total de Ingressantes: 5,010,613
Total de Matriculados: 10,227,266
Total de Concluintes: 1,333,988

DISTRIBUIÇÃO POR MODALIDADE
--------------------------------------------------------------------------------
                      QT_CURSO  QT_VG_TOTAL   QT_ING   QT_MAT  QT_CONC
TP_MODALIDADE_ENSINO                                                  
2                        11297     18590273  3347573  5189391   604742
1                        34479      5075146  1663040  5037875   729246

DISTRIBUIÇÃO POR REGIÃO
--------------------------------------------------------------------------------
              QT_CURSO  QT_VG_TOTAL   QT_ING   QT_MAT  QT_CONC
NO_REGIAO                                                     
Sudeste          14366      2

## Próximos Passos

Possíveis análises futuras:

1. **Análises por Área CINE** - Distribuição de cursos e matrículas por área de conhecimento
2. **Análises por Tipo de Organização Acadêmica** - Universidades vs Faculdades
3. **Análises de Gratuidade** - Cursos gratuitos vs pagos
4. **Análises Demográficas** - Desagregação por sexo, raça/cor em ingressantes e matriculados
5. **Análises de Financiamento** - FIES, ProUni, bolsas
6. **Análises Longitudinais** - Comparação com anos anteriores
7. **Análises por UF** - Distribuição estadual detalhada

Arquivos gerados:
- `resumo_executivo_*.txt`
- `analise_por_regiao.csv`
- `analise_por_modalidade.csv`
- `analise_por_categoria_administrativa.csv`

## 7. Gráfico Dinâmico por Variável

Selecione uma variável numérica para análise estatística e uma dimensão categórica para comparação interativa.

In [11]:
# Grafico dinamico por variavel escolhida
from IPython.display import display
import ipywidgets as widgets

# Variaveis do dicionario para analise (Nome apropriado + codigo)
variaveis_dicionario = {
    'Ano do Censo (NU_ANO_CENSO)': 'NU_ANO_CENSO',
    'UF (SG_UF)': 'SG_UF',
    'Regiao (NO_REGIAO)': 'NO_REGIAO',
    'Nome da IES (NO_IES)': 'NO_IES',
    'Tipo de Rede (TP_REDE)': 'TP_REDE',  # opcional
    'Rotulo CINE (NO_CINE_ROTULO)': 'NO_CINE_ROTULO',
    'Area CINE Geral (NO_CINE_AREA_GERAL)': 'NO_CINE_AREA_GERAL',
    'Modalidade de Ensino (TP_MODALIDADE_ENSINO)': 'TP_MODALIDADE_ENSINO',
    'Quantidade de Cursos (QT_CURSO)': 'QT_CURSO',
    'Vagas Totais (QT_VG_TOTAL)': 'QT_VG_TOTAL',
    'Vagas Totais EAD (QT_VG_TOTAL_EAD)': 'QT_VG_TOTAL_EAD',
    'Ingressantes Total (QT_ING)': 'QT_ING',
    'Ingressantes Feminino (QT_ING_FEM)': 'QT_ING_FEM',
    'Ingressantes Masculino (QT_ING_MASC)': 'QT_ING_MASC',
    'Matriculados Total (QT_MAT)': 'QT_MAT',
    'Matriculados Feminino (QT_MAT_FEM)': 'QT_MAT_FEM',
    'Matriculados Masculino (QT_MAT_MASC)': 'QT_MAT_MASC',
}

# Mapeamento de codigos para exibicao amigavel nos filtros e graficos
value_label_maps = {
    'TP_REDE': {
        1: 'Publica',
        2: 'Privada',
    },
    'TP_MODALIDADE_ENSINO': {
        1: 'Presencial',
        2: 'Curso a distancia',
    },
}

def normalize_key(v):
    if pd.isna(v):
        return None
    try:
        return int(v)
    except Exception:
        return str(v)

def value_label(col, v):
    key = normalize_key(v)
    if col in value_label_maps and key in value_label_maps[col]:
        return f"{value_label_maps[col][key]} ({key})"
    return str(v)

variaveis_disponiveis = {k: v for k, v in variaveis_dicionario.items() if v in df_inep.columns}
col_to_label = {v: k for k, v in variaveis_disponiveis.items()}

if not variaveis_disponiveis:
    print('Nenhuma variavel do dicionario disponivel na base carregada.')
else:
    # Forca colunas codificadas como dimensoes para aparecerem nos filtros
    forced_dimension_cols = {'TP_REDE', 'TP_MODALIDADE_ENSINO'}

    # separa automaticamente variaveis numericas e categoricas com base na base atual
    vars_numericas = {}
    vars_categoricas = {}
    for label, col in variaveis_disponiveis.items():
        if col in forced_dimension_cols:
            vars_categoricas[label] = col
        if pd.api.types.is_numeric_dtype(df_inep[col]):
            vars_numericas[label] = col
        elif pd.api.types.is_string_dtype(df_inep[col]) or pd.api.types.is_categorical_dtype(df_inep[col]) or pd.api.types.is_object_dtype(df_inep[col]):
            vars_categoricas[label] = col

    # fallback de dimensoes para garantir pelo menos 1 opcao
    dims_base = vars_categoricas.copy()
    if 'NO_REGIAO' in variaveis_disponiveis.values() and 'Regiao (NO_REGIAO)' not in dims_base:
        dims_base['Regiao (NO_REGIAO)'] = 'NO_REGIAO'
    if 'SG_UF' in variaveis_disponiveis.values() and 'UF (SG_UF)' not in dims_base:
        dims_base['UF (SG_UF)'] = 'SG_UF'
    for forced_col in forced_dimension_cols:
        if forced_col in variaveis_disponiveis.values():
            forced_label = col_to_label.get(forced_col, forced_col)
            dims_base[forced_label] = forced_col

    available_numeric_vars = vars_numericas
    available_categorical_dims = dims_base

    if not available_numeric_vars:
        print('Sem variaveis numericas para plotar metricas.')
    elif not available_categorical_dims:
        print('Sem dimensoes categoricas para comparacao.')
    else:
        default_var = 'QT_MAT' if 'QT_MAT' in available_numeric_vars.values() else next(iter(available_numeric_vars.values()))
        default_dim = 'NO_REGIAO' if 'NO_REGIAO' in available_categorical_dims.values() else next(iter(available_categorical_dims.values()))

        w_variavel = widgets.Dropdown(
            options=list(available_numeric_vars.items()),
            value=default_var,
            description='Variavel',
            layout=widgets.Layout(width='420px')
        )

        w_dimensao = widgets.Dropdown(
            options=list(available_categorical_dims.items()),
            value=default_dim,
            description='Dimensao',
            layout=widgets.Layout(width='420px')
        )

        w_topn_var = widgets.IntSlider(
            value=10,
            min=5,
            max=30,
            step=1,
            description='Top N',
            continuous_update=False,
            layout=widgets.Layout(width='420px')
        )

        # Filtros adicionais (ate 3)
        no_filter = '__SEM_FILTRO__'
        extra_filter_options = [('Sem filtro', no_filter)] + list(available_categorical_dims.items())

        w_f1_dim = widgets.Dropdown(options=extra_filter_options, value=no_filter, description='Filtro 1', layout=widgets.Layout(width='420px'))
        w_f1_vals = widgets.SelectMultiple(options=[], value=(), description='Valores 1', layout=widgets.Layout(width='420px', height='120px'))

        w_f2_dim = widgets.Dropdown(options=extra_filter_options, value=no_filter, description='Filtro 2', layout=widgets.Layout(width='420px'))
        w_f2_vals = widgets.SelectMultiple(options=[], value=(), description='Valores 2', layout=widgets.Layout(width='420px', height='120px'))

        w_f3_dim = widgets.Dropdown(options=extra_filter_options, value=no_filter, description='Filtro 3', layout=widgets.Layout(width='420px'))
        w_f3_vals = widgets.SelectMultiple(options=[], value=(), description='Valores 3', layout=widgets.Layout(width='420px', height='120px'))

        out_var = widgets.Output()

        def format_series_stats(series: pd.Series) -> pd.DataFrame:
            s = pd.to_numeric(series, errors='coerce').dropna()
            if s.empty:
                return pd.DataFrame()
            stats = {
                'count': s.count(),
                'mean': s.mean(),
                'median': s.median(),
                'std': s.std(),
                'min': s.min(),
                'p25': s.quantile(0.25),
                'p75': s.quantile(0.75),
                'max': s.max(),
                'sum': s.sum(),
            }
            return pd.DataFrame(stats, index=[0]).T.rename(columns={0: 'valor'})

        def fmt_num(value):
            if pd.isna(value):
                return '-'
            return f'{value:,.2f}'.replace(',', 'X').replace('.', ',').replace('X', '.')

        def refresh_filter_values(dim_widget, values_widget):
            selected_col = dim_widget.value
            if selected_col == no_filter:
                values_widget.options = []
                values_widget.value = ()
                return

            vals = df_inep[selected_col].dropna().unique().tolist()
            vals = sorted(vals, key=lambda x: str(x))
            options = [(value_label(selected_col, v), v) for v in vals]
            values_widget.options = options

            valid_values = set(vals)
            current = [v for v in values_widget.value if v in valid_values]
            values_widget.value = tuple(current)

        def apply_extra_filters(data):
            filtered = data
            filters = [
                (w_f1_dim.value, list(w_f1_vals.value)),
                (w_f2_dim.value, list(w_f2_vals.value)),
                (w_f3_dim.value, list(w_f3_vals.value)),
            ]
            for dim_col, selected_vals in filters:
                if dim_col != no_filter and selected_vals:
                    filtered = filtered[filtered[dim_col].isin(selected_vals)]
            return filtered

        def make_dim_label_series(series, dim_col):
            return series.map(lambda v: value_label(dim_col, v))

        def atualizar_grafico_variavel(*_):
            var_col = w_variavel.value
            dim_col = w_dimensao.value
            topn = w_topn_var.value
            var_label = col_to_label.get(var_col, var_col)
            dim_label = col_to_label.get(dim_col, dim_col)

            if var_col not in df_inep.columns or dim_col not in df_inep.columns:
                with out_var:
                    out_var.clear_output(wait=True)
                    print('Variavel ou dimensao nao existe na base carregada.')
                return

            df_work = apply_extra_filters(df_inep)
            if df_work.empty:
                with out_var:
                    out_var.clear_output(wait=True)
                    print('Sem dados apos aplicar os filtros adicionais.')
                return

            df_work = df_work[[var_col, dim_col]].copy()
            df_work[var_col] = pd.to_numeric(df_work[var_col], errors='coerce')
            df_work = df_work.dropna(subset=[var_col, dim_col])

            with out_var:
                out_var.clear_output(wait=True)
                if df_work.empty:
                    print('Sem dados validos para a variavel selecionada.')
                    return

                print(f'Variavel analisada: {var_label}')
                print(f'Dimensao de comparacao: {dim_label}')
                print(f'Registros validos: {len(df_work):,}'.replace(',', '.'))
                print('\nResumo estatistico:')

                stats_df = format_series_stats(df_work[var_col])
                if stats_df.empty:
                    print('Sem estatisticas disponiveis.')
                else:
                    display(stats_df.style.format(fmt_num))

                comp = (
                    df_work.groupby(dim_col, observed=True)[var_col]
                    .agg(['count', 'mean', 'median', 'sum'])
                    .reset_index()
                    .sort_values('sum', ascending=False)
                    .head(topn)
                )

                if comp.empty:
                    print('Sem categorias suficientes para gerar o grafico.')
                    return

                # Copias com labels amigaveis para os eixos
                comp_plot = comp.copy()
                comp_plot['dim_label'] = make_dim_label_series(comp_plot[dim_col], dim_col)
                order_labels = comp_plot['dim_label'].tolist()

                df_plot = df_work[df_work[dim_col].isin(comp[dim_col])].copy()
                df_plot['dim_label'] = make_dim_label_series(df_plot[dim_col], dim_col)

                fig_box = px.box(
                    df_plot,
                    x='dim_label',
                    y=var_col,
                    points='outliers',
                    title=f'Distribuicao de {var_label} por {dim_label}',
                    labels={var_col: var_label, 'dim_label': dim_label},
                    category_orders={'dim_label': order_labels}
                )
                fig_box.update_layout(height=500)
                fig_box.show()

                fig_bar_sum = px.bar(
                    comp_plot,
                    x='dim_label',
                    y='sum',
                    text='sum',
                    title=f'Total absoluto de {var_label} por {dim_label} (Top {topn})',
                    labels={'sum': f'Total de {var_label}', 'dim_label': dim_label},
                    category_orders={'dim_label': order_labels}
                )
                fig_bar_sum.update_traces(texttemplate='%{text:,.0f}', textposition='outside')
                fig_bar_sum.update_layout(height=520)
                fig_bar_sum.show()

                fig_rank = px.bar(
                    comp_plot.sort_values('sum', ascending=True),
                    x='sum',
                    y='dim_label',
                    orientation='h',
                    text='sum',
                    title=f'Ranking de total absoluto de {var_label} por {dim_label}',
                    labels={'sum': f'Total de {var_label}', 'dim_label': dim_label}
                )
                fig_rank.update_traces(texttemplate='%{text:,.0f}', textposition='outside')
                fig_rank.update_layout(height=520)
                fig_rank.show()

        def on_filter_dim_change(change, dim_widget, values_widget):
            if change.get('name') == 'value':
                refresh_filter_values(dim_widget, values_widget)
                atualizar_grafico_variavel()

        w_f1_dim.observe(lambda change: on_filter_dim_change(change, w_f1_dim, w_f1_vals), names='value')
        w_f2_dim.observe(lambda change: on_filter_dim_change(change, w_f2_dim, w_f2_vals), names='value')
        w_f3_dim.observe(lambda change: on_filter_dim_change(change, w_f3_dim, w_f3_vals), names='value')

        for widget_ in [w_variavel, w_dimensao, w_topn_var, w_f1_vals, w_f2_vals, w_f3_vals]:
            widget_.observe(atualizar_grafico_variavel, names='value')

        display(widgets.VBox([
            widgets.HTML('<h4>Grafico Dinamico por Variavel (Dicionario)</h4>'),
            widgets.HBox([w_variavel, w_dimensao]),
            w_topn_var,
            widgets.HTML('<b>Filtros adicionais (ate 3 dimensoes)</b>'),
            widgets.HBox([w_f1_dim, w_f1_vals]),
            widgets.HBox([w_f2_dim, w_f2_vals]),
            widgets.HBox([w_f3_dim, w_f3_vals]),
            out_var
        ]))

        refresh_filter_values(w_f1_dim, w_f1_vals)
        refresh_filter_values(w_f2_dim, w_f2_vals)
        refresh_filter_values(w_f3_dim, w_f3_vals)
        atualizar_grafico_variavel()